In [2]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error
from sklearn.model_selection import  KFold


In [3]:
#Importing the dataset

data= pd.read_excel('data.xlsx')


In [5]:
#Separating the target value

y = data.iloc[:,1].to_numpy()

print(y)
print(y.shape)

[100.         100.         100.         100.          61.78364
  61.78364     61.78364     41.8492      41.8492      41.8492
  39.03435     39.03435     39.03435     38.46022     38.46022
  38.46022     31.07217     31.07217     31.07217     31.07217
  24.76655     24.76655     24.76655    100.         100.
 100.         100.21999    100.21999    100.21999    100.21999
  90.24335184  90.24335184  90.24335184  90.24335184  90.24335184
  85.31762156  85.31762156  85.31762156  56.37816606  56.37816606
  56.37816606  56.37816606  56.37816606  54.32886361  54.32886361
  47.45496411  47.45496411  47.45496411]
(48,)


In [7]:
#Transforming the data using the selected SISSO descriptor

def sisso_transform(X_data: pd.DataFrame):

    d1 = ((X_data["i17de"]/X_data["i14ba"])+(X_data["i28ba"]/X_data["i10de"])) 
    d2 = np.absolute((X_data["i16ba"]*X_data["i16cp"])-(X_data["i19de"]*X_data["i21de"]))


    return d1.to_numpy(), d2.to_numpy()


x1, x2 = sisso_transform(data)

print(x1,x2)

sisso_X = np.column_stack((x1, x2))

print(sisso_X)

[1.06193978 0.98429365 1.01229246 1.03153224 1.28242553 1.23155523
 1.31400808 1.27784866 1.23221055 1.33641722 1.3727913  1.25308541
 1.40352689 1.40432251 1.31823855 1.26983222 1.43888774 1.36325544
 1.39265394 1.39599897 1.41690345 1.48240207 1.41062086 1.03238889
 0.95915344 0.95819447 1.03254177 1.0056872  0.98512635 1.02303125
 1.05755946 1.03171478 1.00380755 1.03203696 1.06797662 1.04158291
 0.9741755  1.00149021 1.18260551 1.18741662 1.17752521 1.19461478
 1.1860428  1.21432224 1.21435034 1.22082402 1.29817774 1.32123739] [27.57236239 22.90743653 21.94011443 22.20455035 16.46172197 13.10693829
 16.17419369  0.65680946  1.91576062  5.35430226  1.57716706  2.71346285
  8.03722522  2.79518342  1.32085328  5.87770052  3.58269715  4.31751189
  4.51411503  2.16376071  2.47343065  5.75822929  0.35886406 20.92681329
 22.5369095  22.23669103 21.64170385 19.88658797 19.43474225 18.32790409
 16.73106841 16.78671588 18.3547644  18.96623521 18.38399432 13.35400144
 17.32772827 19.45726989 

In [8]:
#Performing 5-fold cross-validation

reg = LinearRegression()

y_true_all = []
y_pred_all = []

kf = KFold(n_splits=5, shuffle=True, random_state=42)
folds = kf.split(sisso_X,y)

for train_index, test_index in folds:
    X_train, X_test = sisso_X[train_index], sisso_X[test_index]
    y_train, y_test = y[train_index], y[test_index]


    reg.fit(X_train, y_train)
    y_pred = reg.predict(X_test)

    y_true_all.extend(y_test)
    y_pred_all.extend(y_pred)


y_true_all = np.array(y_true_all)
y_pred_all = np.array(y_pred_all)

print(y_true_all)
print(y_pred_all)


[ 61.78364     39.03435     31.07217    100.         100.
 100.21999    100.21999     85.31762156  56.37816606  54.32886361
 100.          61.78364     41.8492      41.8492      38.46022
  38.46022     31.07217     90.24335184  56.37816606  47.45496411
 100.          61.78364     39.03435     31.07217    100.21999
  90.24335184  90.24335184  90.24335184  90.24335184  47.45496411
 100.         100.          39.03435     24.76655    100.
  85.31762156  85.31762156  56.37816606  54.32886361  41.8492
  38.46022     31.07217     24.76655     24.76655    100.21999
  56.37816606  56.37816606  47.45496411]
[ 64.04235421  38.97149445  30.31142244 104.74921823 104.3616299
  96.28112698  96.05300211  95.76973021  59.7410973   55.43172183
  96.18248508  59.46078459  46.63927956  41.39195154  30.93836552
  48.73182097  37.22322861  91.38684998  55.48277199  53.16079325
 102.29316731  63.03720603  43.87780166  28.56662742  90.48619026
  84.72787145  87.15011201  92.26382007  86.52701907  36.76297202

In [9]:
#Exporting the results

df = pd.DataFrame({
    'CV True': y_true_all,
    'CV Pred': y_pred_all
})

df.to_excel('sisso_predictions.xlsx', index=False)

In [10]:
#Calculating the metrics

r2 = r2_score(y_true_all, y_pred_all)
mae = mean_absolute_error(y_true_all, y_pred_all)
rmse =  np.sqrt(mean_squared_error(y_true_all, y_pred_all))

print(r2)
print(mae)
print(rmse)


0.9673446286808466
4.016006581515185
4.858074465048613
